# 📊 Customer Churn Prediction and Retention Analysis
A Machine Learning project to predict whether a telecom customer will churn using Logistic Regression and Random Forest.

**Dataset:** Telco Customer Churn (7043 rows, 21 columns)  
**Target:** Churn (Yes / No)

## Step 1 - Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries Imported Successfully!")

## Step 2 - Load Dataset

In [ ]:
df = pd.read_csv('data/telecom_churn.csv')

print(f"✅ Dataset Loaded!")
print(f"   Rows    : {df.shape[0]}")
print(f"   Columns : {df.shape[1]}")
df.head()

## Step 3 - Data Exploration

In [ ]:
print("--- Dataset Info ---")
print(df.info())

In [ ]:
print("--- Missing Values ---")
print(df.isnull().sum())

In [ ]:
print("--- Churn Distribution ---")
print(df['Churn'].value_counts())

## Step 4 - Data Cleaning

In [ ]:
# Fix TotalCharges column
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Drop customerID (not useful)
df.drop('customerID', axis=1, inplace=True)

print("✅ Data Cleaning Done!")
df.head()

## Step 5 - Data Visualization

In [ ]:
# Churn Distribution
plt.figure(figsize=(6, 4))
df['Churn'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'], edgecolor='black')
plt.title('Churn Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Churn')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Churn by Internet Service
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='InternetService', hue='Churn', palette=['#2ecc71','#e74c3c'])
plt.title('Churn by Internet Service', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Tenure vs Churn
plt.figure(figsize=(7, 4))
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, palette=['#2ecc71','#e74c3c'])
plt.title('Tenure vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly Charges vs Churn
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette=['#2ecc71','#e74c3c'])
plt.title('Monthly Charges vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Churn by Contract Type
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='Contract', hue='Churn', palette=['#2ecc71','#e74c3c'])
plt.title('Churn by Contract Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 6 - Label Encoding

In [ ]:
le = LabelEncoder()

for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col].astype(str))

print("✅ Label Encoding Done!")
df.head()

## Step 7 - Train / Test Split

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Data Split Done!")
print(f"   Training samples : {X_train.shape[0]}")
print(f"   Testing samples  : {X_test.shape[0]}")

## Step 8 - Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("✅ Feature Scaling Done!")

## Step 9 - Train Models

In [ ]:
# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("✅ Models Trained!")

## Step 10 - Model Evaluation

In [ ]:
print("=" * 40)
print("  Logistic Regression")
print("=" * 40)
print(f"  Accuracy : {accuracy_score(y_test, lr_pred)*100:.2f}%")
print(f"  ROC-AUC  : {roc_auc_score(y_test, lr_pred):.4f}")
print(classification_report(y_test, lr_pred))

In [ ]:
print("=" * 40)
print("  Random Forest")
print("=" * 40)
print(f"  Accuracy : {accuracy_score(y_test, rf_pred)*100:.2f}%")
print(f"  ROC-AUC  : {roc_auc_score(y_test, rf_pred):.4f}")
print(classification_report(y_test, rf_pred))

## Step 11 - Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(confusion_matrix(y_test, lr_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn','Churn'], yticklabels=['No Churn','Churn'], ax=axes[0])
axes[0].set_title(f'Logistic Regression\nAccuracy: {accuracy_score(y_test,lr_pred)*100:.2f}%', fontweight='bold')

sns.heatmap(confusion_matrix(y_test, rf_pred), annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Churn','Churn'], yticklabels=['No Churn','Churn'], ax=axes[1])
axes[1].set_title(f'Random Forest\nAccuracy: {accuracy_score(y_test,rf_pred)*100:.2f}%', fontweight='bold')

plt.tight_layout()
plt.show()

## Step 12 - ROC Curve

In [ ]:
plt.figure(figsize=(7, 5))
for model, name, color in [(lr,'Logistic Regression','#3498db'),(rf,'Random Forest','#e74c3c')]:
    y_prob = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.2f})', color=color, linewidth=2)
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## Step 13 - Feature Importance

In [ ]:
feature_names = df.drop('Churn', axis=1).columns
feat_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(data=feat_df, x='Importance', y='Feature', hue='Feature', palette='viridis', legend=False)
plt.title('Feature Importance - Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

## ✅ Conclusion

| Model | Accuracy |
|---|---|
| Logistic Regression | 81.55% |
| Random Forest | 79.63% |

**Key Findings:**
- Customers with **Fiber optic** internet churn the most
- **Month-to-month** contract customers are more likely to churn
- **Tenure** and **Monthly Charges** are the most important features
- Logistic Regression achieved the best accuracy of **81.55%**